## EDA et Nettoyage gdp


In [506]:
import pandas as pd


In [507]:
df_pib=pd.read_csv('../data economiaque apr api world bank/worldbank_gdp_2015_2025.csv')
df_pib.info()
df_pib.head()
df_pib.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   countryiso3code  2610 non-null   object 
 1   country          2660 non-null   object 
 2   date             2660 non-null   int64  
 3   value            2560 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 83.2+ KB


,date,value
count,2660.000000,2.560000e+03
mean,2019.500000,2.941309e+12
std,2.872821,1.005297e+13
min,2015.000000,3.681194e+07
25%,2017.000000,1.077882e+10
50%,2019.500000,5.988868e+10
75%,2022.000000,6.396812e+11
max,2024.000000,1.109827e+14


### detecter les doublents

In [508]:
print(df_pib.duplicated().sum())


0


### detecter les null

In [509]:
print(df_pib.isnull().sum())


countryiso3code     50
country              0
date                 0
value              100
dtype: int64


### Convetion de Country en dict en suite en obtien juste le nom 

In [510]:
import ast

df_pib["country"] = df_pib["country"].apply(ast.literal_eval)

df_pib["country"] = df_pib["country"].apply(lambda x: x["value"])

### Nettoyages GDP


In [511]:
print(type(df_pib["country"].iloc[0]))

<class 'str'>


### data des pays 

In [512]:
df_iso3=pd.read_csv('../data economiaque apr api world bank/countries_iso3.csv')

### remplire le iso3 avec les nom des pays

In [513]:
df_pib["country"] = df_pib["country"].str.strip()
df_iso3["country_name"] = df_iso3["country_name"].str.strip()

# merge
df = df_pib.merge(df_iso3,
                  left_on="country",
                  right_on="country_name",
                  how="left")

df["countryiso3code"] = df["countryiso3code"].fillna(df["iso3_code"])

df.drop(columns=["country_name", "iso3_code"], inplace=True)

print(df["countryiso3code"].isnull().sum())

0


### supprimer les regions


In [514]:
aggregates = [
    "AFE","AFW","AFR","ARB","CEB","CSS","EAP",
    "EAR","EAS","ECA","ECS","EMU","EUU",
    "FCS","HIC","IBD","IBT","IDA",
    "IDB","IDX","LAC","LCN","LDC",
    "LIC","LMC","LMY","LTE","MEA",
    "MIC","MNA","NAC","OED","PRE",
    "PSS","PST","SAS","SSA","SSF",
    "UMC","WLD",'AND', 'ASM', 'BMU', 'CHI', 'CUB', 'ERI', 'FRO', 'GIB', 'GRL',
       'GUM', 'IMN', 'INX', 'LIE', 'MAF', 'MCO', 'MHL', 'MNP', 'NRU',
       'PRI', 'PRK', 'PYF', 'SOM', 'TCA', 'TKM', 'TUV', 'VGB', 'VIR',
       'YEM'
]

df = df[~df["countryiso3code"].isin(aggregates)]

print(df["countryiso3code"].nunique())
print(df.isnull().sum())

199
countryiso3code     0
country             0
date                0
value              18
dtype: int64


In [515]:
print(df[df["countryiso3code"].str.len() != 3])

Empty DataFrame
Columns: [countryiso3code, country, date, value]
Index: []


In [517]:
df["date"] = df["date"].astype(int)
df = df.sort_values(["countryiso3code", "date"])

df["value"] = df.groupby("countryiso3code")["value"] \
                .transform(lambda x: x.fillna(x.median()))

print(df["value"].isnull().sum())

0


In [518]:
print(df.isnull().sum())

countryiso3code    0
country            0
date               0
value              0
dtype: int64


#### -----------------------------------------------------------------------------------------------------------------------------

### EDA GDP per capita

In [519]:
df_pibCa=pd.read_csv('../data economiaque apr api world bank/worldbank_gdp_per_capita_2015_2025.csv')
df_pibCa.info()
df_pibCa.head()
df_pibCa.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   countryiso3code  2610 non-null   object 
 1   country          2660 non-null   object 
 2   date             2660 non-null   int64  
 3   value            2560 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 83.2+ KB


,date,value
count,2660.000000,2560.000000
mean,2019.500000,17736.756126
std,2.872821,26716.630587
min,2015.000000,219.424831
25%,2017.000000,2321.266604
50%,2019.500000,6747.230038
75%,2022.000000,22334.524273
max,2024.000000,288001.433369


### les doublents

In [520]:
print(df_pibCa.duplicated().sum())

0


### les null

In [521]:
print(df_pibCa.isnull().sum())

countryiso3code     50
country              0
date                 0
value              100
dtype: int64


### garder juste les nom des pays

In [522]:
import ast

df_pibCa["country"] = df_pibCa["country"].apply(ast.literal_eval)

df_pibCa["country"] = df_pibCa["country"].apply(lambda x: x["value"])

### remplir le iso3

In [523]:
df_pibCa["country"] = df_pibCa["country"].str.strip()
df_iso3["country_name"] = df_iso3["country_name"].str.strip()

# merge
df2 = df_pibCa.merge(df_iso3,
                  left_on="country",
                  right_on="country_name",
                  how="left")

df2["countryiso3code"] = df2["countryiso3code"].fillna(df2["iso3_code"])

df2.drop(columns=["country_name", "iso3_code"], inplace=True)

print(df2["countryiso3code"].isnull().sum())

0


In [525]:
print(df2.isnull().sum())

countryiso3code      0
country              0
date                 0
value              100
dtype: int64


### supprimer les region et les ils

In [526]:
aggregates = [
    "AFE","AFW","AFR","ARB","CEB","CSS","EAP",
    "EAR","EAS","ECA","ECS","EMU","EUU",
    "FCS","HIC","IBD","IBT","IDA",
    "IDB","IDX","LAC","LCN","LDC",
    "LIC","LMC","LMY","LTE","MEA",
    "MIC","MNA","NAC","OED","PRE",
    "PSS","PST","SAS","SSA","SSF",
    "UMC","WLD",'AND', 'ASM', 'BMU', 'CHI', 'CUB', 'ERI', 'FRO', 'GIB', 'GRL',
       'GUM', 'IMN', 'INX', 'LIE', 'MAF', 'MCO', 'MHL', 'MNP', 'NRU',
       'PRI', 'PRK', 'PYF', 'SOM', 'TCA', 'TKM', 'TUV', 'VGB', 'VIR',
       'YEM'
]

df2 = df2[~df2["countryiso3code"].isin(aggregates)]

print(df2["countryiso3code"].nunique())
print(df2.isnull().sum())

199
countryiso3code     0
country             0
date                0
value              18
dtype: int64


### remplire les null par le median

In [527]:
df2["date"] = df2["date"].astype(int)
df2 = df2.sort_values(["countryiso3code", "date"])

df2["value"] = df2.groupby("countryiso3code")["value"] \
                .transform(lambda x: x.fillna(x.median()))

print(df2["value"].isnull().sum())

0


# EDA & Nettoyage Inflation

In [528]:
df_Infla=pd.read_csv('../data economiaque apr api world bank/worldbank_inflation_2015_2025.csv')
df_Infla.info()
df_Infla.head()
df_Infla.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   countryiso3code  2610 non-null   object 
 1   country          2660 non-null   object 
 2   date             2660 non-null   int64  
 3   value            2289 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 83.2+ KB


,date,value
count,2660.000000,2289.000000
mean,2019.500000,6.552725
std,2.872821,21.949032
min,2015.000000,-12.296984
25%,2017.000000,1.482999
50%,2019.500000,3.093421
75%,2022.000000,5.856838
max,2024.000000,557.201817


### les doublents

In [529]:
df_Infla.duplicated().sum()

0

### les nulls

In [530]:
df_Infla.isnull().sum()

countryiso3code     50
country              0
date                 0
value              371
dtype: int64

### remplissage des country vide

In [531]:
df_Infla["country"] = df_Infla["country"].apply(ast.literal_eval)

df_Infla["country"] = df_Infla["country"].apply(lambda x: x["value"])

In [532]:
df_Infla["country"] = df_Infla["country"].str.strip()
df_iso3["country_name"] = df_iso3["country_name"].str.strip()

# merge
df3= df_Infla.merge(df_iso3,
                  left_on="country",
                  right_on="country_name",
                  how="left")

df3["countryiso3code"] = df3["countryiso3code"].fillna(df3["iso3_code"])

df3.drop(columns=["country_name", "iso3_code"], inplace=True)

print(df3["countryiso3code"].isnull().sum())

0


### suppression des region et des ils

In [533]:
aggregates = [
    "AFE","AFW","AFR","ARB","CEB","CSS","EAP",
    "EAR","EAS","ECA","ECS","EMU","EUU",
    "FCS","HIC","IBD","IBT","IDA",
    "IDB","IDX","LAC","LCN","LDC",
    "LIC","LMC","LMY","LTE","MEA",
    "MIC","MNA","NAC","OED","PRE",
    "PSS","PST","SAS","SSA","SSF",
    "UMC","WLD"
]

df3 = df3[~df3["countryiso3code"].isin(aggregates)]

print(df3["countryiso3code"].nunique())
print(df3.isnull().sum())

227
countryiso3code      0
country              0
date                 0
value              371
dtype: int64


## remplir les null de l inflation par le median

In [534]:
df3["date"] = df3["date"].astype(int)
df3 = df3.sort_values(["countryiso3code", "date"])

df3["value"] = df3.groupby("countryiso3code")["value"] \
                .transform(lambda x: x.fillna(x.median()))

print(df3["value"].isnull().sum())

280


c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Us

In [535]:
df3[df3["value"].isnull()]["countryiso3code"].unique()

array(['AND', 'ASM', 'BMU', 'CHI', 'CUB', 'ERI', 'FRO', 'GIB', 'GRL',
       'GUM', 'IMN', 'INX', 'LIE', 'MAF', 'MCO', 'MHL', 'MNP', 'NRU',
       'PRI', 'PRK', 'PYF', 'SOM', 'TCA', 'TKM', 'TUV', 'VGB', 'VIR',
       'YEM'], dtype=object)

## on supprime ces pays Ces pays représentent des cas structurellement différents — leur absence n'affecte pas 

In [536]:

pays_sans_data = df3[df3["value"].isnull()]["countryiso3code"].unique()
df3 = df3[~df3["countryiso3code"].isin(pays_sans_data)]

print("Nulls restants:", df3["value"].isnull().sum())  # doit afficher 0
print("Pays restants:", df3["countryiso3code"].nunique())

Nulls restants: 0
Pays restants: 199


### EDA & Nettooyage de la population

In [537]:
df_Pop=pd.read_csv('../data economiaque apr api world bank/worldbank_population_2015_2025.csv')
df_Pop.info()
df_Pop.head()
df_Pop.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   countryiso3code  2610 non-null   object 
 1   country          2660 non-null   object 
 2   date             2660 non-null   int64  
 3   value            2650 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 83.2+ KB


,date,value
count,2660.000000,2.650000e+03
mean,2019.500000,3.164976e+08
std,2.872821,9.784742e+08
min,2015.000000,9.646000e+03
25%,2017.000000,1.786911e+06
50%,2019.500000,1.054135e+07
75%,2022.000000,6.453566e+07
max,2024.000000,8.141809e+09


### les Doublents & les nulls

In [538]:
df_Pop.duplicated().sum()
df_Pop.isnull().sum()

countryiso3code    50
country             0
date                0
value              10
dtype: int64

### remplir le iso3

In [539]:
df_Pop["country"] = df_Pop["country"].apply(ast.literal_eval)

df_Pop["country"] = df_Pop["country"].apply(lambda x: x["value"])

In [540]:
df_Pop["country"] = df_Pop["country"].str.strip()
df_iso3["country_name"] = df_iso3["country_name"].str.strip()

# merge
df4= df_Pop.merge(df_iso3,
                  left_on="country",
                  right_on="country_name",
                  how="left")

df4["countryiso3code"] = df4["countryiso3code"].fillna(df4["iso3_code"])

df4.drop(columns=["country_name", "iso3_code"], inplace=True)

print(df4["countryiso3code"].isnull().sum())

0


### supp des region et des ils

In [541]:
aggregates = [
    "AFE","AFW","AFR","ARB","CEB","CSS","EAP",
    "EAR","EAS","ECA","ECS","EMU","EUU",
    "FCS","HIC","IBD","IBT","IDA",
    "IDB","IDX","LAC","LCN","LDC",
    "LIC","LMC","LMY","LTE","MEA",
    "MIC","MNA","NAC","OED","PRE",
    "PSS","PST","SAS","SSA","SSF",
    "UMC","WLD",'AND', 'ASM', 'BMU', 'CHI', 'CUB', 'ERI', 'FRO', 'GIB', 'GRL',
       'GUM', 'IMN', 'INX', 'LIE', 'MAF', 'MCO', 'MHL', 'MNP', 'NRU',
       'PRI', 'PRK', 'PYF', 'SOM', 'TCA', 'TKM', 'TUV', 'VGB', 'VIR',
       'YEM'
]

df4 = df4[~df4["countryiso3code"].isin(aggregates)]

print(df4["countryiso3code"].nunique())
print(df4.isnull().sum())

199
countryiso3code    0
country            0
date               0
value              0
dtype: int64


### fusionee lesindecateurs

In [ ]:
# Renommer correctement (sans espace)
df_gdp    = df[["countryiso3code", "date", "value"]].rename(columns={"value": "gdp"})
df_gdpCap = df2[["countryiso3code", "date", "value"]].rename(columns={"value": "gdp_per_capita"})
df_infla  = df3[["countryiso3code", "date", "value"]].rename(columns={"value": "inflation"})
df_pop    = df4[["countryiso3code", "date", "value"]].rename(columns={"value": "population"})

# Fusionner
df_macro = df_gdp \
    .merge(df_gdpCap, on=["countryiso3code", "date"], how="inner") \
    .merge(df_infla,  on=["countryiso3code", "date"], how="inner") \
    .merge(df_pop,    on=["countryiso3code", "date"], how="inner")

print("Shape:", df_macro.shape)
print("Colonnes:", df_macro.columns.tolist())
print(df_macro.head())

Shape: (1990, 6)
Colonnes: ['countryiso3code', 'date', 'gdp', 'gdp_per_capita', 'inflation', 'population']
  countryiso3code  date           gdp  gdp_per_capita  inflation  population
0             ABW  2015  2.962907e+09    27458.220154   0.474764    107906.0
1             ABW  2016  2.983637e+09    27441.550214  -0.931196    108727.0
2             ABW  2017  3.092428e+09    28440.041688  -1.028282    108735.0
3             ABW  2018  3.276188e+09    30082.158423   3.626041    108908.0
4             ABW  2019  3.346623e+09    30645.890602   4.257462    109203.0


### souvgarder en csv


In [543]:
Economique_data= df_macro.to_csv('../data economiaque apr api world bank/economique_data.csv', index=False)

# EDA & Nettoyade de data d agricol

In [544]:
df_agri1 = pd.read_csv('../data agriculture/agricol_prix2015_2018.csv')   # ← sans underscore
df_agri2 = pd.read_csv('../data agriculture/agricol_prix_2019_2021.csv')
df_agri3 = pd.read_csv('../data agriculture/agricol_prix_2022_2024.csv')


### detection des nombre de ligne , les anees desponible , les nulls et boublons

In [545]:
df_agri = pd.concat([df_agri1, df_agri2, df_agri3], ignore_index=True)

# Vérification
print("Shape total:", df_agri.shape)
print("Années disponibles:", sorted(df_agri["Year"].unique()))
print("Nulls:\n", df_agri.isnull().sum())
print("Doublons:", df_agri.duplicated().sum())
print(df_agri.info())

Shape total: (75755, 14)
Années disponibles: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Nulls:
 Domain Code         0
Domain              0
Area Code (M49)     0
Area                0
Element Code        0
Element             0
Item Code (CPC)     0
Item                0
Year Code           0
Year                0
Unit                0
Value               0
Flag                0
Flag Description    0
dtype: int64
Doublons: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75755 entries, 0 to 75754
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Domain Code       75755 non-null  object
 1   Domain            75755 non-null  object
 2   Area Code (M49)   75755 non-null  int64 
 3   Area              75755 non-null  object
 4   Element Code      75755 non-null  int64 
 5   Element           75755 non-null  object
 6   Item Code (CPC)   75755 non-null  object
 7   Item              75755 n

### souvgarde des colonnes principales

In [ ]:
# Garder uniquement les colonnes importantes
df_agri = df_agri[["Area", "Item", "Year", "Value"]]

# Renommer pour plus de clarté
df_agri.rename(columns={
    "Area"             : "country",
    "Item"             : "product",
    "Year"             : "date",
    "Value"            : "gross_production_value",
}, inplace=True)

print(df_agri.shape)
print(df_agri.head())

(75755, 4)
   country   product  date  gross_production_value
0  Albania    Apples  2015                   38075
1  Albania    Apples  2016                   42140
2  Albania    Apples  2017                   39984
3  Albania    Apples  2018                   44980
4  Albania  Apricots  2015                    3762


In [559]:
df_agri.to_csv('../data agriculture/agricol_prix_clean.csv', index=False)

### EDA & Nettoyage  des prix


In [547]:
df_Idice = pd.read_csv('../data_prix_et_indice_produit/Consumer_indece.csv')   # ← sans underscore

### data consumer indice

### detection des nombre de ligne , les anees desponible , les nulls et boublons

In [548]:
print("Shape total:", df_Idice.shape)
print("Années disponibles:", sorted(df_Idice["Year"].unique()))
print("Nulls:\n", df_Idice.isnull().sum())
print("Doublons:", df_Idice.duplicated().sum())

Shape total: (25676, 17)
Années disponibles: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Nulls:
 Domain Code             0
Domain                  0
Area Code (M49)         0
Area                    0
Year Code               0
Year                    0
Item Code               0
Item                    0
Months Code             0
Months                  0
Element Code            0
Element                 0
Unit                25676
Value                   0
Flag                    0
Flag Description        0
Note                    0
dtype: int64
Doublons: 0


In [ ]:
# Vérifier les colonnes réelles
print(df_Idice.columns.tolist())

# Sélectionner les bonnes colonnes (Item = produit alimentaire)
df_Idice = df_Idice[["Area", "Item", "Year", "Element", "Value"]]

# Renommer
df_Idice.rename(columns={
    "Area"    : "country",
    "Item"    : "product",       
    "Year"    : "date",
    "Element" : "indicator",    
    "Value"   : "Consumer_indice",
}, inplace=True)

print(df_Idice.shape)
print(df_Idice.head())

['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Year Code', 'Year', 'Item Code', 'Item', 'Months Code', 'Months', 'Element Code', 'Element', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']
(25676, 5)
       country                                     product  date indicator  \
0  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
1  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
2  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
3  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
4  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   

   Consumer_indice  
0       102.473425  
1       100.599141  
2        99.388333  
3        99.195658  
4        99.034873  


In [550]:
print(df_Idice.columns.tolist())
print(df_Idice.head())

['country', 'product', 'date', 'indicator', 'Consumer_indice']
       country                                     product  date indicator  \
0  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
1  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
2  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
3  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   
4  Afghanistan  Consumer Prices, Food Indices (2015 = 100)  2015     Value   

   Consumer_indice  
0       102.473425  
1       100.599141  
2        99.388333  
3        99.195658  
4        99.034873  


In [551]:
df_Idice = df_Idice.groupby(["country", "date"], as_index=False)["Consumer_indice"].mean()

In [556]:
df_Idice.to_csv('../data_prix_et_indice_produit/Consumer_indice_clean.csv', index=False)

### data de price producer

In [552]:
import csv

def parse_price_producer(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        for outer_row in reader:
            if not outer_row: continue
            inner_parsed = list(csv.reader([outer_row[0]]))[0]
            if len(inner_parsed) >= 16:
                rows.append(inner_parsed[:16])
    df = pd.DataFrame(rows, columns=header)
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    df['Year']  = pd.to_numeric(df['Year'],  errors='coerce')
    return df

# Ancien fichier (format cassé) → parser spécial
df_Price_new = parse_price_producer('../data_prix_et_indice_produit/Price_Producer.csv')

#  Nouveau fichier (format normal) → pandas direct
df_Price_old = pd.read_csv('../data_prix_et_indice_produit/producer_price2015.2018.csv')
df_Price_old['Value'] = pd.to_numeric(df_Price_old['Value'], errors='coerce')
df_Price_old['Year']  = pd.to_numeric(df_Price_old['Year'],  errors='coerce')

# Garder mêmes colonnes avant concat
df_Price_old = df_Price_old[["Area", "Item", "Year", "Value"]]
df_Price_new = df_Price_new[["Area", "Item", "Year", "Value"]]

# Concatener
df_Price = pd.concat([df_Price_old, df_Price_new], ignore_index=True)
df_Price.rename(columns={
    "Area"  : "country",
    "Item"  : "product",
    "Year"  : "date",
    "Value" : "producer_price_usd"
}, inplace=True)

# Vérification
print("Shape:", df_Price.shape)
print("Années:", sorted(df_Price['date'].dropna().unique().astype(int)))
print("Nb pays:", df_Price['country'].nunique())
print(df_Price.head())

Shape: (48753, 4)
Années: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Nb pays: 148
   country   product  date  producer_price_usd
0  Albania    Apples  2015               389.0
1  Albania    Apples  2016               381.3
2  Albania    Apples  2017               461.6
3  Albania    Apples  2018               435.2
4  Albania  Apricots  2015               746.3


In [553]:
# Voir le format exact du nouveau fichier
with open('../data_prix_et_indice_produit/producer_price2015.2018.csv', 'r', encoding='utf-8-sig') as f:
    for i, line in enumerate(f):
        print(repr(line[:200]))
        if i >= 3: break

'Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Months Code,Months,Unit,Value,Flag,Flag Description\n'
'"PP","Producer Prices","008","Albania","5532","Producer Price (USD/tonne)","01341","Apples","2015","2015","7021","Annual value","USD","389","A","Official figure"\n'
'"PP","Producer Prices","008","Albania","5532","Producer Price (USD/tonne)","01341","Apples","2016","2016","7021","Annual value","USD","381.3","A","Official figure"\n'
'"PP","Producer Prices","008","Albania","5532","Producer Price (USD/tonne)","01341","Apples","2017","2017","7021","Annual value","USD","461.6","A","Official figure"\n'


### detection des nombre de ligne , les anees desponible , les nulls et boublons

In [554]:
print("Shape total:", df_Price.shape)
print("Années disponibles:", sorted(df_Price["date"].unique()))
print("Nulls:\n", df_Price.isnull().sum())
print("Doublons:", df_Price.duplicated().sum())
print(df_Price.info())

Shape total: (48753, 4)
Années disponibles: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Nulls:
 country               0
product               0
date                  0
producer_price_usd    0
dtype: int64
Doublons: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48753 entries, 0 to 48752
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             48753 non-null  object 
 1   product             48753 non-null  object 
 2   date                48753 non-null  int64  
 3   producer_price_usd  48753 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.5+ MB
None


In [557]:
df_Price.to_csv('../data_prix_et_indice_produit/producer_price_clean.csv', index=False)

## EDA & Nettoyage de la data de meteo


In [555]:
# Lire les 2 fichiers
df_meteo1 = pd.read_csv('../meteo/AllCountries_2015_2022.csv')
df_meteo2 = pd.read_csv('../meteo/AllCountries_2023_2025.csv')

# Concatener
df_meteo = pd.concat([df_meteo1, df_meteo2], ignore_index=True)

# Extraire l'année
df_meteo['year'] = df_meteo['year_month'].str[:4].astype(int)

# Agréger par pays + année
df_meteo_annual = df_meteo.groupby(['countryName', 'year'], as_index=False).agg(
    tavg = ('tavg', 'mean'),
    prcp = ('prcp', 'sum'),
    tmax = ('tmax', 'mean'),
    tmin = ('tmin', 'mean'),
)

# ✅ Vérification
print("Shape:", df_meteo_annual.shape)
print("Années:", sorted(df_meteo_annual['year'].unique()))
print("Nb pays:", df_meteo_annual['countryName'].nunique())
print("Nulls:\n", df_meteo_annual.isnull().sum())
print(df_meteo_annual.head())

Shape: (2533, 6)
Années: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Nb pays: 247
Nulls:
 countryName     0
year            0
tavg            0
prcp            0
tmax           12
tmin           10
dtype: int64
   countryName  year       tavg   prcp       tmax       tmin
0  Afghanistan  2015  17.814474    0.0  23.350594  12.385103
1  Afghanistan  2016  19.053654    0.3  24.033464  12.902735
2  Afghanistan  2017  18.747540  284.5  25.550561  13.418746
3  Afghanistan  2018  18.308045  163.0  25.147071  12.019698
4  Afghanistan  2019  17.348956  974.1  23.544207  11.214021


In [558]:
df_meteo_annual.to_csv('../meteo/annual_meteo_clean.csv', index=False)